In [20]:
import Create_Target_Variable
import pandas as pd
import re

In [14]:
# gather data
plays_df = pd.DataFrame()

plays_files = ['2017_plays.csv',
               '2018_plays.csv',
               '2019_plays.csv',
               '2020_plays.csv',
               '2021_plays.csv',
               '2022_plays.csv',
               '2023_plays.csv',
               '2024_plays.csv',
               '2025_plays.csv'
               ]

for file in plays_files:
    temp_df = pd.read_csv('Classify Plays/' + file)
    plays_df = pd.concat([plays_df, temp_df])

In [15]:
# drop rows where PlayStart is null - irrelevant data
plays_df.dropna(subset=['PlayStart'], inplace=True)

# drop sacks and penalty plays - can't classify a play call
plays_df = plays_df[~plays_df['PlayOutcome'].str.contains('sack|penalty', case=False, na=False)]

In [16]:
# create target variable: play call
plays_df['PlayCall'] = plays_df.apply(Create_Target_Variable.classify_play_call, axis=1)

In [ ]:
# drop where play cannot be classified
plays_df.dropna(subset=['PlayCall'], inplace=True)

In [35]:
# map team with possesion to abbreviated name
team_name_map = {
    'Arizona Cardinals': 'ARI',
    'Dallas Cowboys': 'DAL',
    'Houston Texans': 'HOU',
    'Carolina Panthers': 'CAR',
    'Minnesota Vikings': 'MIN',
    'Buffalo Bills': 'BUF',
    'Miami Dolphins': 'MIA',
    'Atlanta Falcons': 'ATL',
    'Washington Commanders': 'WAS',
    'Baltimore Ravens': 'BAL',
    'Jacksonville Jaguars': 'JAX',
    'New England Patriots': 'NE',
    'Chicago Bears': 'CHI',
    'Denver Broncos': 'DEN',
    'New Orleans Saints': 'NO',
    'Cleveland Browns': 'CLE',
    'Green Bay Packers': 'GB',
    'Philadelphia Eagles': 'PHI',
    'Pittsburgh Steelers': 'PIT',
    'New York Giants': 'NYG',
    'Tampa Bay Buccaneers': 'TB',
    'Cincinnati Bengals': 'CIN',
    'Kansas City Chiefs': 'KC',
    'San Francisco 49ers': 'SF',
    'New York Jets': 'NYJ',
    'Tennessee Titans': 'TEN',
    'Los Angeles Rams': 'LAR',
    'Las Vegas Raiders': 'LV',
    'Detroit Lions': 'DET',
    'Indianapolis Colts': 'IND',
    'Los Angeles Chargers': 'LAC',
    'Seattle Seahawks': 'SEA'
}

plays_df['TeamWithPossessionShort'] = plays_df['TeamWithPossession'].map(team_name_map)

In [ ]:
### extract data from PlayStart field e.g. "3rd & 1 at LAC 1" ###

# split before and after " at "
temp = plays_df['PlayStart'].str.split(' at ', expand=True)
left = temp[0]   # "3rd & 1"
right = temp[1]  # "LAC 1"

# extract Down and YdsTo1stDown
plays_df['Down'] = left.str.extract(r'(\d+)').astype(int)
plays_df['YdsTo1stDown'] = left.str.extract(r'&\s*(\d+)').astype(int)

# extract Territory and YdPosition
plays_df['Territory'] = right.str.extract(r'([A-Z]+)')
plays_df['YdPosition'] = right.str.extract(r'(\d+)').astype(int)

In [36]:
# convert JAC to JAX and OAK to LV in Territory field
plays_df['Territory'] = plays_df['Territory'].replace({
    'JAC': 'JAX',
    'OAK': 'LV'
})

In [38]:
# calculate yards to enzone
def get_yds_to_endzone(row):
    if row['YdPosition'] == 50:
        return 50
    elif row['Territory'] == row['TeamWithPossessionShort']:
        return row['YdPosition'] + 50
    else:
        return row['YdPosition']

plays_df['YdsToEndzone'] = plays_df.apply(get_yds_to_endzone, axis=1)